Setup + imports

In [46]:
import sys
from pathlib import Path
import pandas as pd
import plotly.express as px
import numpy as np


ROOT = Path.cwd().parent
if str(ROOT / "src") not in sys.path:
    sys.path.append(str(ROOT / "src"))


from bootcamp_data.config import make_paths
pa = make_paths(ROOT)

Load + Audit

In [45]:
df = pd.read_parquet(pa.processed / "analytics_table.parquet")


print(f"Total Rows: {len(df):,}")


print("\nColumn Data Types (First 15) ")
print(df.dtypes.head(15))


print("\nMissing Values Report")
missing_report = df.isna().sum().sort_values(ascending=False)
print(missing_report[missing_report > 0])

Total Rows: 5

Column Data Types (First 15) 
order_id               string[python]
user_id                string[python]
amount                        Float64
quantity                        Int64
created_at        datetime64[ns, UTC]
status                         object
status_clean           string[python]
amount__isna                     bool
quantity__isna                   bool
date                           object
year                          float64
month                  string[python]
day                            object
hour                          float64
country                        object
dtype: object

Missing Values Report
quantity             1
amount               1
amount_is_outlier    1
amount_winsor        1
created_at           1
date                 1
hour                 1
day                  1
month                1
year                 1
dtype: int64


Question 1: What is the Daily Revenue?

In [ ]:
daily_trend = (
    df.groupby('date', dropna=False)['amount']
    .sum()
    .reset_index()
    .rename(columns={'amount': 'revenue'})
)


fig = px.line(daily_trend, x='date', y='revenue', 
              title="Daily Revenue Trend",
              markers=True)


fig.write_image(pa.figures / "daily_revenue.png")


Question 2: Which countries are the top revenue ?

In [ ]:
country_rev = (
    df.groupby('country', dropna=False)['amount']
    .sum()
    .reset_index()
    .sort_values('amount', ascending=False)
    .head(10)
)


fig = px.bar(country_rev, x='country', y='amount', 
             title="Top  Countries by Revenue",
             color='country')


fig.write_image(pa.figures / "revenue_by_country.png")

Question 3: What is the Distribution of Order Statuses?

In [59]:
status_dist = df['status_clean'].value_counts().reset_index()

fig = px.pie(status_dist, names='status_clean', values='count', 
             title="Distribution of Order Statuses",
             color='status_clean')

fig.write_image(pa.figures / "status_distribution.png")


Bootstrap comparison

In [61]:
def bootstrap_diff_means(a: pd.Series, b: pd.Series, *, n_boot: int = 2000, seed: int = 0) -> dict:
    rng = np.random.default_rng(seed)
    a = pd.to_numeric(a, errors="coerce").dropna().to_numpy()
    b = pd.to_numeric(b, errors="coerce").dropna().to_numpy()
    assert len(a) > 0 and len(b) > 0, "Empty group after cleaning"

    diffs = []
    for _ in range(n_boot):
        sa = rng.choice(a, size=len(a), replace=True)
        sb = rng.choice(b, size=len(b), replace=True)
        diffs.append(sa.mean() - sb.mean())
    diffs = np.array(diffs)

    return {
        "diff_mean": float(a.mean() - b.mean()),
        "ci_low": float(np.quantile(diffs, 0.025)),
        "ci_high": float(np.quantile(diffs, 0.975)),
    }

d = df.assign(is_refund=df["status_clean"].eq("refund").astype(int))
a = d.loc[d["country"].eq("SA"), "is_refund"]
b = d.loc[d["country"].eq("AE"), "is_refund"]
print("n_SA:", len(a), "n_AE:", len(b))
print(bootstrap_diff_means(a, b, n_boot=2000, seed=0))

n_SA: 4 n_AE: 1
{'diff_mean': -1.0, 'ci_low': -1.0, 'ci_high': -1.0}
